<a href="https://colab.research.google.com/github/YizhongHu/Symm4ML/blob/main/lie_companion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lie Groups — Companion Notebook

**6.7970/8.750 Symmetry and its Application to Machine Learning**

This notebook follows the Lie Groups exercise section by section.
Use it to **prototype your code** and **test your implementations**
against the course library before submitting on the website.

Each section includes small tests you can use to check your work.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atomicarchitects/symm4ml-colabs/blob/main/lie_companion.ipynb)

## Setup

In [1]:
%%capture
!pip install https://symm4ml.mit.edu/_static/symm4ml_s26/symm4ml/symm4ml_latest.zip

In [2]:
import numpy as np
from numpy import einsum as ein
from typing import List

from symm4ml import groups, linalg, rep, lie

### Reference data

These structure constants and generators are used throughout the exercise for testing.

In [3]:
# SO(3) structure constants
so3_A = np.array(
    [
        [[0.0, 0.0, 0.0], [0.0, 0.0, 1.0], [0.0, -1.0, 0.0]],
        [[0.0, 0.0, -1.0], [0.0, 0.0, 0.0], [1.0, 0.0, 0.0]],
        [[0.0, 1.0, 0.0], [-1.0, 0.0, 0.0], [0.0, 0.0, 0.0]],
    ]
)

# SO(3) L=1 generators
so3_X = np.array(
    [
        [[0.0, 0.0, 0.0], [0.0, 0.0, -1.0], [0.0, 1.0, 0.0]],
        [[0.0, 0.0, 1.0], [0.0, 0.0, 0.0], [-1.0, 0.0, 0.0]],
        [[0.0, -1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 0.0]],
    ]
)

# SO(1,3) structure constants
so13_A = np.array(
    [
        [
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
            [0.0, -1.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
            [0.0, 0.0, 0.0, 0.0, -1.0, 0.0],
        ],
        [
            [0.0, 0.0, -1.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, -1.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 1.0, 0.0, 0.0],
        ],
        [
            [0.0, 1.0, 0.0, 0.0, 0.0, 0.0],
            [-1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 1.0, 0.0],
            [0.0, 0.0, 0.0, -1.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
            [0.0, 0.0, 0.0, 0.0, -1.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, -1.0, 0.0, 0.0, 0.0],
            [0.0, 1.0, 0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 0.0, 0.0, -1.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 1.0, 0.0, 0.0],
            [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [-1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 0.0, 1.0, 0.0],
            [0.0, 0.0, 0.0, -1.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, -1.0, 0.0, 0.0, 0.0, 0.0],
            [1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        ],
    ]
)

# SO(1,3) generators
so13_X = np.array(
    [
        [
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, -1.0],
            [0.0, 0.0, 1.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 1.0],
            [0.0, 0.0, 0.0, 0.0],
            [0.0, -1.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, -1.0, 0.0],
            [0.0, 1.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 1.0, 0.0, 0.0],
            [1.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 1.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
            [1.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 1.0],
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
            [1.0, 0.0, 0.0, 0.0],
        ],
    ]
)

# SU(2) generators
su2_X = np.array(
    [
        [
            [0.0 + 0.0j, 0.5 + 0.0j],
            [-0.5 + 0.0j, 0.0 + 0.0j],
        ],
        [
            [-0.0 - 0.5j, 0.0 + 0.0j],
            [0.0 + 0.0j, 0.0 + 0.5j],
        ],
        [
            [0.0 - 0.0j, 0.0 + 0.5j],
            [0.0 + 0.5j, 0.0 - 0.0j],
        ],
    ]
)

print(f"so3_A: {so3_A.shape}, so3_X: {so3_X.shape}")
print(f"so13_A: {so13_A.shape}, so13_X: {so13_X.shape}")
print(f"su2_X: {su2_X.shape}")

so3_A: (3, 3, 3), so3_X: (3, 3, 3)
so13_A: (6, 6, 6), so13_X: (6, 4, 4)
su2_X: (3, 2, 2)


---
## 1. `are_isomorphic(X1, X2)`

Check if two representations of a Lie algebra (expressed in terms of generators) are isomorphic, i.e., the same up to a similarity transform. You can use a function from a previous problem set.

In [4]:
def are_isomorphic(X1: np.ndarray, X2: np.ndarray, *, tol: float = 1e-8) -> bool:
    """Checks if two representations of a Lie group are isomorphic.
    Input:
        X1: np.array [lie_dim, d1, d1] - generators of a representation.
        X2: np.array [lie_dim, d2, d2] - generators of a representation.
    Output:
        are_isomorphic: bool - True if the representations are isomorphic.
    """
    return len(linalg.infer_change_of_basis(X1, X2)) > 0

In [5]:
# Small tests from lie.py
assert are_isomorphic(so3_X, so3_X)
print("are_isomorphic tests passed!")

are_isomorphic tests passed!


---
## 2. `tensor_product(X1, X2)`

Compute the tensor product of two representations of a Lie algebra, where both input and output are expressed in terms of generators.

In [6]:
def tensor_product(X1: np.ndarray, X2: np.ndarray) -> np.ndarray:
    """Tensor product of two representations of a Lie group.
    Input:
        X1: np.array [lie_dim, d1, d1] - generators of a representation.
        X2: np.array [lie_dim, d2, d2] - generators of a representation.
    Output:
        X: np.array [lie_dim, d1*d2, d1*d2] - tensor product of the representations.
    """
    X = []
    d1, d2 = X1.shape[1], X2.shape[1]
    for mat1, mat2 in zip(X1, X2):
        X.append(np.kron(mat1, np.eye(d2)) + np.kron(np.eye(d1), mat2))
    return np.array(X)

In [7]:
# Small tests from lie.py
assert tensor_product(so3_X, so3_X).shape == (3, 9, 9)
print("tensor_product tests passed!")

tensor_product tests passed!


---
## 3. `is_a_representation(algebra, X)`

Check whether the given generators `X` satisfy the commutation relations encoded in `algebra`: $[X_i, X_j] = \sum_k A_{ijk} X_k$.

In [8]:
def is_a_representation(
    algebra: np.ndarray, X: np.ndarray, *, tol: float = 1e-8
) -> bool:
    """Check if X satisfies the commutation relations of the Lie algebra:
        [X_i, X_j] = sum_k A_{ijk} X_k
    Input:
        algebra: np.array [lie_dim, lie_dim, lie_dim] - Lie algebra (structure constants)
        X: np.array [lie_dim, d, d] - generators of a representation.
    Output:
        is_a_representation: bool - True if X satisfies the commutation relations.
    """
    lie_dim = algebra.shape[0]
    for i in range(lie_dim):
        for j in range(lie_dim):
          Xi_Xj = X[i] @ X[j] - X[j] @ X[i]
          sum_Xk = np.sum(algebra[i, j].reshape(-1, 1, 1) * X, axis=0)
          if np.allclose(Xi_Xj, sum_Xk, atol=tol):
            continue
          else:
            return False
    return True

In [9]:
# Small tests from lie.py
assert is_a_representation(so3_A, so3_X)
print("is_a_representation tests passed!")

is_a_representation tests passed!


---
## 4. `is_an_irrep(algebra, X)`

Check if a representation of a Lie algebra is irreducible. Return `True` if the input is a valid representation AND is irreducible.

In [10]:
def is_an_irrep(algebra: np.ndarray, X: np.ndarray, *, tol: float = 1e-8) -> bool:
    """Checks if X is an irreducible representation of the Lie algebra.
    Input:
        algebra: np.array [lie_dim, lie_dim, lie_dim] - Lie algebra (structure constants)
        X: np.array [lie_dim, d, d] - generators of a representation.
    Output:
        is_an_irrep: bool - True if X is an irreducible representation.
    """
    return is_a_representation(algebra, X) and len(linalg.infer_change_of_basis(X, X)) == 1

In [11]:
# Small tests from lie.py
assert is_an_irrep(so3_A, so3_X)
print("is_an_irrep tests passed!")

is_an_irrep tests passed!


---
## 5. `decompose_rep_into_irreps(X)`

Decompose a representation of a Lie algebra into a direct sum of irreducible representations. Follow the same algorithm as for finite groups.

In [12]:
def decompose_rep_into_irreps(X: np.array, *, tol: float = 1e-8) -> List[np.array]:
    """Decomposes representation into irreducible representations.
    Input:
        X: np.array [lie_dim, d, d] - generators of a representation.
    Output:
        Ys: List[np.array] - list of generators of irreducible representations.
    """
    Q = linalg.infer_change_of_basis(X, X, tol=tol)
    alpha = np.random.rand(Q.shape[0])
    alpha /= np.linalg.norm(alpha)
    Q_bar = np.sum(alpha.reshape(-1, 1, 1) * Q, axis=0)
    assert Q_bar.shape == (X.shape[1], X.shape[2])
    val, vec = np.linalg.eigh(Q_bar)
    Ys = []
    for _, vecs in linalg.eigenspaces(val, vec, tol=tol):
        # G-S requires vectors to be row-major
        B, _ = linalg.gram_schmidt(vecs.T, tol=tol)
        # B is row-major so we flip them
        Ys.append(np.array([
            B.conj() @ mat @ B.T for mat in X
        ]))
    return Ys

In [13]:
# Small tests from lie.py
assert {
    X.shape[1] for X in decompose_rep_into_irreps(tensor_product(so3_X, so3_X))
} == {1, 3, 5}
print("decompose_rep_into_irreps tests passed!")

decompose_rep_into_irreps tests passed!


---
## 6. `infer_irreps_from_tensor_products(X, n)`

Infer `n` non-isomorphic finite-dimensional irreps of the underlying matrix Lie group by successively decomposing tensor products. Start with the trivial representation and take successive tensor products with `X` and (if needed) its conjugate `X*`.

In [42]:
import scipy as sp

def infer_irreps_from_tensor_products(
    X: np.ndarray, n: int, *, tol: float = 1e-8
) -> List[np.ndarray]:
    """Infers irreducible representations from successive tensor products of a representation.
    Input:
        X: np.array [lie_dim, d, d] - generators of a representation.
        n: int - number of non-isomorphic irreducible representations to infer.
    Output:
        Ys: List[np.array] - list of n generators of irreducible representations,
            each an np.array of shape [lie_dim, d', d'] for some d'.
    """
    lie_dim = X.shape[0]
    irreps = [np.array([[[0.0]]] * lie_dim)]
    pointer = 0
    while len(irreps) < n:
        new_irreps = decompose_rep_into_irreps(tensor_product(irreps[pointer], X))
        for new_irrep in new_irreps:
            if all(len(linalg.infer_change_of_basis(irrep, new_irrep, tol=tol)) == 0 for irrep in irreps):
                irreps.append(new_irrep)
        if len(irreps) >= n:
            break

        new_irreps = decompose_rep_into_irreps(tensor_product(irreps[pointer], X.conj()))
        for new_irrep in new_irreps:
            if all(len(linalg.infer_change_of_basis(irrep, new_irrep, tol=tol)) == 0 for irrep in irreps):
                irreps.append(new_irrep)
        pointer += 1
    return irreps[:n]

In [43]:
# Small tests from lie.py
Xs = infer_irreps_from_tensor_products(so3_X, 3)
assert len(Xs) == 3
assert Xs[0].shape == (3, 1, 1)
assert is_an_irrep(so3_A, Xs[0])
assert Xs[1].shape == (3, 3, 3)
assert is_an_irrep(so3_A, Xs[1])
assert Xs[2].shape == (3, 5, 5)
assert is_an_irrep(so3_A, Xs[2])
print("infer_irreps_from_tensor_products tests passed!")

infer_irreps_from_tensor_products tests passed!


---
## Explore Further

In [47]:
# Try inferring irreps of SO(1,3) or SU(2)!
infer_irreps_from_tensor_products(su2_X, 5)

[array([[[0.]],
 
        [[0.]],
 
        [[0.]]]),
 array([[[ 0. +0.j ,  0.5+0.j ],
         [-0.5+0.j ,  0. +0.j ]],
 
        [[ 0. -0.5j,  0. +0.j ],
         [ 0. +0.j ,  0. +0.5j]],
 
        [[ 0. +0.j ,  0. +0.5j],
         [ 0. +0.5j,  0. +0.j ]]]),
 array([[[ 9.80544826e-18+0.j        , -8.22056720e-01+0.j        ,
           5.57788186e-01+0.j        ],
         [ 8.22056720e-01+0.j        ,  8.80627025e-20+0.j        ,
          -1.14433776e-01+0.j        ],
         [-5.57788186e-01+0.j        ,  1.14433776e-01+0.j        ,
          -2.57166130e-20+0.j        ]],
 
        [[ 0.00000000e+00+0.02328219j,  0.00000000e+00+0.15018497j,
           0.00000000e+00+0.01860224j],
         [ 0.00000000e+00+0.15018497j,  0.00000000e+00+0.9109387j ,
           0.00000000e+00+0.354314j  ],
         [ 0.00000000e+00+0.01860224j,  0.00000000e+00+0.354314j  ,
           0.00000000e+00-0.93422089j]],
 
        [[ 0.00000000e+00-0.2261689j ,  0.00000000e+00-0.53420057j,
           0.0000